# Data Analysis

Hanan Safeer | Waseem Shaikh

Created:       May 20, 2026

Last Updated:  May 20, 2026

### Importing Required Libraries

In [109]:
import pandas as pd
import numpy as np

### Loading Cleaned Dataset

In [110]:
# Load cleaned dataset
df = pd.read_csv("../data/silver/online_retail_cleaned.csv", parse_dates=["InvoiceDate"])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Cancelled,ProductCategory,TotalPrice,Year,Month,Day,Hour,Weekend,ExpenseCategory
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,Decor,15.30,2010,12,1,8,False,Medium
1,536365,71053,WHITE MOROCCAN METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,Lighting,20.34,2010,12,1,8,False,Medium
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,Decor,22.00,2010,12,1,8,False,Medium
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,Kitchen,20.34,2010,12,1,8,False,Medium
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,Decor,20.34,2010,12,1,8,False,Medium


### Basic Dataset Overview

In [111]:
# Shape
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Data types
df.info()

# Summary statistics
df.describe()

# Unique counts
print("Unique Customers:", df['CustomerID'].nunique())
print("Unique Products:", df['StockCode'].nunique())
print("Countries:", df['Country'].nunique())
print(df['Country'].value_counts().head())

Rows: 535278
Columns: 17
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 535278 entries, 0 to 535277
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   InvoiceNo        535278 non-null  object        
 1   StockCode        535278 non-null  object        
 2   Description      535278 non-null  object        
 3   Quantity         535278 non-null  float64       
 4   InvoiceDate      535278 non-null  datetime64[ns]
 5   UnitPrice        535278 non-null  float64       
 6   CustomerID       535278 non-null  int64         
 7   Country          535278 non-null  object        
 8   Cancelled        535278 non-null  bool          
 9   ProductCategory  535278 non-null  object        
 10  TotalPrice       535278 non-null  float64       
 11  Year             535278 non-null  int64         
 12  Month            535278 non-null  int64         
 13  Day              535278 non-null  int64         


### Extra Cleaning Steps

1. Replacing Country Abbreviations: 

In [112]:
# Replace EIRE and RSA with full country names
df["Country"] = df["Country"].replace({
    "EIRE": "Ireland",
    "RSA": "South Africa"
})

df["Country"].value_counts().head(10)

Country
United Kingdom    488937
Germany             9480
France              8541
Ireland             8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: count, dtype: int64

2. Deleting Records with Invalid Country Names:

In [113]:
invalid_countries = ["European Community", "Unspecified", "Channel Islands"]

df = df[~df["Country"].isin(invalid_countries)].copy()

df["Country"].value_counts().head(10)

Country
United Kingdom    488937
Germany             9480
France              8541
Ireland             8184
Spain               2528
Netherlands         2371
Belgium             2069
Switzerland         1994
Portugal            1510
Australia           1258
Name: count, dtype: int64

3. Deleting Records with Product Category - Other

In [114]:
df = df[df["ProductCategory"] != "Other"].copy()
df["ProductCategory"].value_counts()

ProductCategory
Decor          137498
Kitchen         58263
Accessories     34705
Home            12616
Lighting         1514
Office            945
Name: count, dtype: int64

### Delete All Records Where CustomerID = -1

In [115]:
df = df[df["CustomerID"] != -1].copy()

### Loading the Cleaned Dataframe to a CSV File

In [116]:
df.to_csv("../data/gold/final_cleaned_data.csv", index=False)

# Core Analysis

### Monthly Revenue

In [117]:
monthly_revenue = (
    df.groupby(["Year", "Month", "Country", "ProductCategory"])
      ["TotalPrice"]
      .sum()
      .reset_index()
)

monthly_revenue.head(13)

,Year,Month,Country,ProductCategory,TotalPrice
0,2010,12,Australia,Accessories,19.50
1,2010,12,Australia,Decor,248.75
2,2010,12,Australia,Home,99.30
3,2010,12,Australia,Kitchen,69.65
4,2010,12,Austria,Decor,52.16
5,2010,12,Austria,Kitchen,79.60
6,2010,12,Belgium,Accessories,122.29
7,2010,12,Belgium,Decor,411.89
8,2010,12,Belgium,Home,15.90
9,2010,12,Belgium,Kitchen,374.50


In [118]:
monthly_revenue.to_csv("../data/gold/monthly_revenue.csv", index=False)


### Country Revenue

In [119]:
country_revenue = (
    df.groupby(["Country", "ProductCategory", "StockCode", "Description"])
      ["TotalPrice"]
      .sum()
      .reset_index()
)

country_revenue.head()


,Country,ProductCategory,StockCode,Description,TotalPrice
0,Australia,Accessories,20711,JUMBO BAG TOYS,179.00
1,Australia,Accessories,20712,JUMBO BAG WOODLAND ANIMALS,61.10
2,Australia,Accessories,20717,STRAWBERRY SHOPPER BAG,12.50
3,Australia,Accessories,21056,DOCTOR'S BAG SOFT TOY,26.85
4,Australia,Accessories,21577,SAVE THE PLANET COTTON TOTE BAG,27.00


In [120]:

country_revenue.to_csv("../data/gold/country_revenue.csv", index=False)

### Product Revenue

In [121]:
product_revenue = (
    df.groupby(["Country", "ProductCategory", "StockCode", "Description"])
      .agg(
          TotalQuantity=("Quantity", "sum"),
          TotalRevenue=("TotalPrice", "sum")
      )
      .reset_index()
)

product_revenue.head()

,Country,ProductCategory,StockCode,Description,TotalQuantity,TotalRevenue
0,Australia,Accessories,20711,JUMBO BAG TOYS,100.0,179.00
1,Australia,Accessories,20712,JUMBO BAG WOODLAND ANIMALS,30.0,61.10
2,Australia,Accessories,20717,STRAWBERRY SHOPPER BAG,10.0,12.50
3,Australia,Accessories,21056,DOCTOR'S BAG SOFT TOY,3.0,26.85
4,Australia,Accessories,21577,SAVE THE PLANET COTTON TOTE BAG,12.0,27.00


In [122]:
product_revenue.to_csv("../data/gold/product_revenue.csv", index=False)

### Category Revenue

In [123]:
category_revenue = (
    df.groupby("ProductCategory")["TotalPrice"]
      .sum()
      .reset_index()
)

category_revenue.head()

,ProductCategory,TotalPrice
0,Accessories,619885.59
1,Decor,2209113.09
2,Home,337865.94
3,Kitchen,1115318.71
4,Lighting,26862.72


In [124]:
category_revenue.to_csv("../data/gold/category_revenue.csv", index=False)

### Hourly Sales

In [125]:
hourly_sales = (
    df.groupby("Hour")["TotalPrice"]
      .sum()
      .reset_index()
)

hourly_sales.head(25)

,Hour,TotalPrice
0,6,121.45
1,7,16521.06
2,8,151596.58
3,9,339004.78
4,10,609219.23
5,11,547443.73
6,12,661062.78
7,13,590113.18
8,14,495495.49
9,15,461852.20


In [126]:
hourly_sales.to_csv("../data/gold/hourly_sales.csv", index=False)

### Weekend Sales

In [127]:
weekend_sales = (
    df.groupby("Weekend")["TotalPrice"]
      .sum()
      .reset_index()
)


weekend_sales.head()

,Weekend,TotalPrice
0,False,3931672.69
1,True,389569.77


In [128]:
weekend_sales.to_csv("../data/gold/weekend_sales.csv", index=False)

# Customer Analysis

### Customer Value Table

In [129]:
customer_value = (
    df.groupby("CustomerID")
      .agg(
          TotalRevenue=("TotalPrice", "sum"),
          OrderCount=("InvoiceNo", "nunique"),
          TotalQuantity=("Quantity", "sum")
      )
      .reset_index()
)

customer_value["AOV"] = customer_value["TotalRevenue"] / customer_value["OrderCount"]
customer_value["RevenueRank"] = customer_value["TotalRevenue"].rank(pct=True)

def segment_customer(p):
    if p >= 0.80:
        return "High Value"
    elif p >= 0.20:
        return "Mid Value"
    else:
        return "Low Value"

customer_value["Segment"] = customer_value["RevenueRank"].apply(segment_customer)

customer_value.head()


,CustomerID,TotalRevenue,OrderCount,TotalQuantity,AOV,RevenueRank,Segment
0,12347,1764.08,7,1220.0,252.011429,0.898860,High Value
1,12348,586.76,3,1040.0,195.586667,0.671652,Mid Value
2,12349,801.40,1,334.0,801.400000,0.742403,Mid Value
3,12350,20.40,1,24.0,20.400000,0.019706,Low Value
4,12352,662.89,8,279.0,82.861250,0.700142,Mid Value


In [130]:
customer_value.to_csv("../data/gold/customer_value.csv", index=False)

### Customer Orders Table

In [131]:
customer_orders = (
    df.groupby(["CustomerID", "InvoiceNo"])
      .agg(
          OrderRevenue=("TotalPrice", "sum"),
          OrderQuantity=("Quantity", "sum"),
          OrderDate=("InvoiceDate", "min")
      )
      .reset_index()
)

customer_orders.head()

,CustomerID,InvoiceNo,OrderRevenue,OrderQuantity,OrderDate
0,12347,537626,185.05,147.0,2010-12-07 14:57:00
1,12347,542237,221.49,167.0,2011-01-26 14:30:00
2,12347,549222,215.35,189.0,2011-04-07 10:43:00
3,12347,556201,180.70,82.0,2011-06-09 13:01:00
4,12347,562032,253.95,165.0,2011-08-02 08:48:00


In [132]:
customer_orders.to_csv("../data/gold/customer_orders.csv", index=False)

### Customer Country Table 

In [133]:
customer_country = (
    df.groupby(["CustomerID", "Country"])
      .agg(
          TotalRevenue=("TotalPrice", "sum"),
          OrderCount=("InvoiceNo", "nunique")
      )
      .reset_index()
)

customer_country.head()

,CustomerID,Country,TotalRevenue,OrderCount
0,12347,Iceland,1764.08,7
1,12348,Finland,586.76,3
2,12349,Italy,801.40,1
3,12350,Norway,20.40,1
4,12352,Norway,662.89,8


In [134]:
customer_country.to_csv("../data/gold/customer_country.csv", index=False)

### Customer Activity Table

In [135]:
customer_activity = (
    df.groupby(["CustomerID", "Hour", "Day", "Weekend"])
      .agg(
          Orders=("InvoiceNo", "nunique"),
          Revenue=("TotalPrice", "sum")
      )
      .reset_index()
)

customer_activity.head(15)

,CustomerID,Hour,Day,Weekend,Orders,Revenue
0,12347,8,2,False,1,253.95
1,12347,10,7,False,1,215.35
2,12347,12,31,False,1,685.76
3,12347,13,9,False,1,180.70
4,12347,14,7,False,1,185.05
5,12347,14,26,False,1,221.49
6,12347,15,7,False,1,21.78
7,12348,10,5,False,1,17.00
8,12348,10,25,False,1,62.16
9,12348,19,16,False,1,507.60


In [136]:
customer_activity.to_csv("../data/gold/customer_activity.csv", index=False)

# Time Series Analysis

### Daily Revenue

In [137]:
daily_revenue = (
    df.groupby(df["InvoiceDate"].dt.date)["TotalPrice"]
      .sum()
      .reset_index(name="TotalRevenue")
)

daily_revenue.head()


,InvoiceDate,TotalRevenue
0,2010-12-01,20423.18
1,2010-12-02,21864.02
2,2010-12-03,13548.04
3,2010-12-05,15003.13
4,2010-12-06,13167.56


In [138]:
daily_revenue.to_csv("../data/gold/daily_revenue.csv", index=False)

###  Weekly Revenue

In [139]:
df["Week"] = df["InvoiceDate"].dt.isocalendar().week

weekly_revenue = (
    df.groupby(["Year", "Week", "Country", "ProductCategory"])
      ["TotalPrice"]
      .sum()
      .reset_index()
)

weekly_revenue.head()


,Year,Week,Country,ProductCategory,TotalPrice
0,2010,48,Australia,Decor,169.1
1,2010,48,Belgium,Accessories,28.0
2,2010,48,Belgium,Decor,38.4
3,2010,48,Belgium,Kitchen,223.9
4,2010,48,France,Accessories,79.2


In [140]:
weekly_revenue.to_csv("../data/gold/weekly_revenue.csv", index=False)

### Seasonality Table

In [141]:
seasonality = (
    df.groupby(["Month", "Day", "Hour"])["TotalPrice"]
      .sum()
      .reset_index()
)

seasonality.head(20)


,Month,Day,Hour,TotalPrice
0,1,4,10,938.68
1,1,4,11,829.67
2,1,4,12,854.15
3,1,4,13,674.56
4,1,4,14,1400.44
5,1,4,15,804.35
6,1,4,16,321.25
7,1,5,9,283.50
8,1,5,10,2519.71
9,1,5,11,4438.79


In [142]:
seasonality.to_csv("../data/gold/seasonality.csv", index=False)

# RFM Segmentation

### RFM Segments & RFM Scores

In [143]:
# --- RFM BASE TABLE ---

# Snapshot date = day after last transaction
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_scores = (
    df.groupby("CustomerID")
      .agg(
          Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
          Frequency=("InvoiceNo", "nunique"),
          Monetary=("TotalPrice", "sum")
      )
      .reset_index()
)

# --- R, F, M RANKS (FIXED VERSION) ---

# Recency: lower days = better → labels reversed
rfm_scores["R_rank"] = pd.qcut(
    rfm_scores["Recency"],
    4,
    labels=[4, 3, 2, 1]
)

# Frequency: use rank to avoid duplicate bin edges
rfm_scores["F_rank"] = pd.qcut(
    rfm_scores["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

# Monetary: same trick as Frequency
rfm_scores["M_rank"] = pd.qcut(
    rfm_scores["Monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

# --- RFM SCORE ---

rfm_scores["RFM_Score"] = (
    rfm_scores["R_rank"].astype(int)
    + rfm_scores["F_rank"].astype(int)
    + rfm_scores["M_rank"].astype(int)
)

# --- OPTIONAL: SEGMENT LABELS ---

def rfm_segment(score):
    if score >= 10:
        return "Champion"
    elif score >= 8:
        return "Loyal"
    elif score >= 6:
        return "Potential Loyalist"
    elif score >= 4:
        return "At Risk"
    else:
        return "Hibernating"

rfm_scores["Segment"] = rfm_scores["RFM_Score"].apply(rfm_segment)
rfm_scores.head()


,CustomerID,Recency,Frequency,Monetary,R_rank,F_rank,M_rank,RFM_Score,Segment
0,12347,2,7,1764.08,4,4,4,12,Champion
1,12348,249,3,586.76,1,3,3,7,Potential Loyalist
2,12349,19,1,801.40,3,1,3,7,Potential Loyalist
3,12350,310,1,20.40,1,1,1,3,Hibernating
4,12352,36,8,662.89,3,4,3,10,Champion


In [144]:
# --- SAVE TO GOLD ---

# Full RFM scores table
rfm_scores.to_csv("../data/gold/rfm_scores.csv", index=False)